# Proof of concept: Adding Spatial Wavelet Decomposition and Reconstruction to Chromatin Deconvolution
February 16, 2024

We've shown that it is possible to decompose a 2D image into wavelet coefficients and reconstruct within cvxpy. The next step is the feasibility with doing a large set of image coefficients and with the chromatin deconvolution. We had previously deconvolved the coefficients of the images, but were unable to enforce valid non-negative coefficients that produce images. So, this may be a chance to find a place where we can enforce valid non-negative f images, because we are now able to reconstruct the images from coefficients within cvxpy to enforce constraints or norms on the minimization function. We will also be checking feasibility and timing of introducing these additional wavelet computations.

In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
import numpy as np
import pywt

In [3]:
# let's start with a 16x16 of a chromatin image

from cc_src.chromatin_model import ChromatinModel
from src.config import load_yl_replicate1_rg1_alpha_vst_config

config = load_yl_replicate1_rg1_alpha_vst_config()
chromatin_model = ChromatinModel(config)
chromatin_model.load_mnase_gene('CLN1', replicate=1)

Loading MNase reads for CLN1...Done.


In [4]:
n = len(chromatin_model.timepoints)

exact_bins = chromatin_model.create_exact_bins()
G_images = chromatin_model.downsample_bins(exact_bins, 25, 32, 288, 512)
G = G_images.reshape((n, -1))
image_shape = G_images[0].shape

print("The shape of G is: ", G.shape)
print("The shape of G images is: ", G_images.shape)

The shape of G is:  (16, 256)
The shape of G images is:  (16, 8, 32)


In [5]:
chromatin_model.setup_deconv_model()

In [6]:
from src.deconvolve_chromatin import deconvolve_chromatin_H

deconv_model = chromatin_model.deconv_model
deconv_model.gamma = 0
H = deconv_model.H

In [7]:
G_images.shape

(16, 8, 32)

In [8]:
import pywt
from src.wavelets_2d_linalg import wave2d_decomposition, wave2d_reconstruction, create_wavelet2d_convolution_matrices

image = G_images[0]
image_shape = image.shape
n = G_images.shape[0]

wavelet = pywt.Wavelet('bior2.2')
(decomp_mats, recon_mats) = create_wavelet2d_convolution_matrices(wavelet, image_shape)

for i in range(n):
    image = G_images[i]
    coeffs = wave2d_decomposition(image, decomp_mats)
    reconstructed = wave2d_reconstruction(coeffs, recon_mats)


In [9]:

# Create a deconvolution function that attemps to find the best coefficients
# that when reconstructed into F_images
#
# (H@F_images / G - 1) is minimized
#
#
# Therefore, we will need to be able to reconstruct the images from coefficients
# in the optimization function
coeffs[0].shape

(4, 16)

In [33]:
from src.deconvolve_wavelet_chromatin import deconvolve_wavelet_chromatin
coeffs_shape = coeffs[0].shape
f = deconvolve_wavelet_chromatin(deconv_model, H, G, coeffs_shape,
    verbose=True, allow_negative=True, image_shape=image_shape)


Exception: Cannot evaluate the truth value of a constraint or chain constraints, e.g., 1 >= x >= 0.